# Week 5: Linear algebra as geometry (vectors, matrices, eigenvectors, PCA)

**Budget:** about 10 hours. **Month 2 rule:** every concept goes intuition → paper → numbers → NumPy → where it shows up in ML.

| Session | Topic | Time |
|---|---|---|
| Day 1 | Vectors, dot products, cosine similarity, vectorised NumPy | 2.5 h |
| Day 2 | Matrices as transformations; solving $Ax = b$; determinants | 2.5 h |
| Day 3 | Eigenvalues and eigenvectors: what $\lambda$ *means*; power iteration | 2.5 h |
| Day 4 | PCA from scratch on a medical dataset | 2 h |
| Review | Spaced recall and a blank-page rebuild | 0.5 h |

**Watch first** (about 1 h in total, spread over the week): 3Blue1Brown, *Essence of Linear Algebra*, chapters 1–4 (before Day 2) and chapter 14 (before Day 3).

### Done when
- [ ] Every test cell prints ✅.
- [ ] You can find the eigenvalues of a 2×2 matrix **by hand**, and explain eigenvectors without formulas.
- [ ] Your `pca` matches scikit-learn (up to the sign of each component).
- [ ] You can explain why vectorised NumPy is faster than Python loops.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

---
# Day 1: Vectors and similarity (2.5 h)

A vector is a list of numbers, but think of it as an **arrow** from the origin. In AI, *everything* becomes a vector: a document (word counts), an image (its pixels), a patient record (its features), a word (its embedding).

The **dot product** $u \cdot v = \sum_i u_i v_i = \|u\|\,\|v\|\cos\theta$ measures how much two arrows point the same way. Dividing out the lengths gives **cosine similarity**, which is the backbone of vector search and of the retrieval half of your capstone.

In [ ]:
u = np.array([3.0, 4.0])
v = np.array([4.0, 3.0])
print("dot", u @ v, "| norms", np.linalg.norm(u), np.linalg.norm(v))
print("cos", (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v)))

# document vectors: counts of the words [pneumonia, xray, diet]
doc_a = np.array([3, 2, 0])
doc_b = np.array([6, 4, 0])      # the same topic, twice as long
doc_c = np.array([0, 0, 5])
for name, d in [("b", doc_b), ("c", doc_c)]:
    print(name, "dot:", doc_a @ d)

✍️ *doc_b has a bigger dot product with doc_a just because it's longer. Why does cosine fix this, and why does that matter for search?* (double-click to edit)



### Exercise 5.1: `cosine`
Return the cosine similarity of two 1-D arrays. Return `0.0` if either vector has zero length.

In [ ]:
def cosine(u, v):
    raise NotImplementedError

In [ ]:
assert abs(cosine(np.array([1, 0]), np.array([0, 1]))) < 1e-12
assert abs(cosine(np.array([3, 2, 0]), np.array([6, 4, 0])) - 1) < 1e-12
assert cosine(np.array([0, 0]), np.array([1, 1])) == 0.0
print("✅ 5.1 passed")

### Vectorisation: no Python loops over data
NumPy runs loops in compiled C. One matrix operation can replace millions of Python-level steps. **Broadcasting** stretches arrays with size-1 dimensions so their shapes match:

In [ ]:
A = np.arange(6).reshape(2, 3)       # shape (2, 3)
row = np.array([10, 20, 30])         # shape (3,), treated as (1, 3)
col = np.array([[1], [2]])           # shape (2, 1)
print(A + row)
print(A * col)
print("norm of every row:", np.linalg.norm(A, axis=1))

### Exercise 5.2: `cosine_matrix`
Given `A` with shape (n, d) and `B` with shape (m, d), return the (n, m) matrix of cosine similarities between every row of A and every row of B, **without Python loops**. Treat zero rows as having cosine 0. This is exactly what a vector search engine does with one query batch against a whole corpus.

In [ ]:
def cosine_matrix(A, B):
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
A, B = rng.normal(size=(50, 8)), rng.normal(size=(40, 8))
C = cosine_matrix(A, B)
assert C.shape == (50, 40)
assert abs(C[3, 7] - cosine(A[3], B[7])) < 1e-10
A[0] = 0
assert np.all(cosine_matrix(A, B)[0] == 0)

import time
A, B = rng.normal(size=(300, 64)), rng.normal(size=(300, 64))
t0 = time.perf_counter(); slow = [[cosine(a, b) for b in B] for a in A]; t1 = time.perf_counter()
fast = cosine_matrix(A, B); t2 = time.perf_counter()
assert np.allclose(slow, fast)
print(f"loops {t1 - t0:.3f}s, vectorised {t2 - t1:.4f}s")
print("✅ 5.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Normalise first: divide each row by its norm (`np.linalg.norm(A, axis=1, keepdims=True)`). Then the cosine is just a dot product.

</details>

<details><summary>Hint 2</summary>

All pairwise dot products of rows: `A_normalised @ B_normalised.T`.

</details>

<details><summary>Hint 3</summary>

To avoid dividing by zero, use `np.maximum(norms, 1e-12)`. Zero rows then become zero vectors, and their cosines are 0.

</details>


---
# Day 2: Matrices are transformations (2.5 h)

A matrix $A$ is a **function** that moves every vector: $x \mapsto Ax$. Its **columns** tell you where the basis vectors $e_1 = (1, 0)$ and $e_2 = (0, 1)$ end up. Everything else follows by linearity. Run the demo and watch what happens to the grid and the unit circle.

In [ ]:
def show_transform(A, title):
    t = np.linspace(0, 2 * np.pi, 100)
    circle = np.stack([np.cos(t), np.sin(t)])
    grid = [np.stack([np.full(20, x), np.linspace(-1, 1, 20)]) for x in np.linspace(-1, 1, 5)]
    grid += [g[::-1] for g in grid]
    fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
    for ax, M, name in [(axes[0], np.eye(2), "before"), (axes[1], A, "after")]:
        for g in grid:
            p = M @ g; ax.plot(p[0], p[1], color="lightgray")
        c = M @ circle; ax.plot(c[0], c[1], "b")
        e1, e2 = M @ np.array([1, 0]), M @ np.array([0, 1])
        ax.arrow(0, 0, *e1, color="r", width=0.02); ax.arrow(0, 0, *e2, color="g", width=0.02)
        ax.set_aspect("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_title(f"{title}: {name}")
    plt.show()

theta = np.pi / 6
show_transform(np.array([[2, 0], [0, 0.5]]), "scale")
show_transform(np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]]), "rotate")
show_transform(np.array([[1, 1], [0, 1]]), "shear")
show_transform(np.array([[1, 2], [0.5, 1]]), "singular")

✍️ *For each of the 4 matrices: what happened to the circle? For the singular one, why did the plane collapse to a line, and what is its determinant?* (double-click to edit)



**Matrix multiplication = composition.** $(AB)x = A(Bx)$: first apply $B$, then $A$. A neural network layer computes $Wx + b$, a transformation, and the activation function then bends the result (Week 13).

### Exercise 5.3: `matmul` with loops
Implement matrix multiplication with three nested loops: $C_{ij} = \sum_k A_{ik} B_{kj}$. Raise `ValueError` if the shapes don't fit. (You'll never use this in practice, but writing it once makes shape rules stick.)

In [ ]:
def matmul(A, B):
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(1)
A, B = rng.normal(size=(3, 4)), rng.normal(size=(4, 2))
assert np.allclose(matmul(A, B), A @ B)
try:
    matmul(A, A); raise AssertionError("should raise")
except ValueError:
    pass
print("✅ 5.3 passed")

### Solving $Ax = b$
"Which input $x$ does $A$ send to $b$?" When $\det A \neq 0$, there's exactly one answer. The **determinant** is the factor by which $A$ scales areas; $\det = 0$ means space gets squashed, so the information can't be recovered. Use `np.linalg.solve`, **never** `inv(A) @ b` (it's slower and less accurate).

In [ ]:
A = np.array([[2.0, 1.0], [1.0, 3.0]])
b = np.array([3.0, 5.0])
x = np.linalg.solve(A, b)
print("x =", x, "check:", A @ x, "det:", np.linalg.det(A))

### Exercise 5.4: Linear regression is solving a linear system
To fit $y \approx Xw$, the least-squares solution satisfies the **normal equations** $X^\top X w = X^\top y$. Write `fit_linear(X, y)` using `np.linalg.solve`. (In Week 6 you'll get the same answer by gradient descent.)

In [ ]:
def fit_linear(X, y):
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(2)
X = np.c_[np.ones(100), rng.normal(size=(100, 2))]     # first column of ones = intercept
w_true = np.array([1.0, 2.0, -3.0])
y = X @ w_true + 0.01 * rng.normal(size=100)
w = fit_linear(X, y)
assert np.allclose(w, w_true, atol=0.01)
assert np.allclose(w, np.linalg.lstsq(X, y, rcond=None)[0])
print("✅ 5.4 passed:", w)

---
# Day 3: Eigenvalues and eigenvectors (2.5 h)

Most vectors get *knocked off their direction* by a matrix. An **eigenvector** is a special direction that the matrix only **stretches**:
$$A v = \lambda v$$
The **eigenvalue** $\lambda$ is the stretch factor: $\lambda = 2$ doubles the vector, $\lambda = 0.5$ halves it, and a negative $\lambda$ flips it.

**Why it matters in ML:** the eigenvectors of a covariance matrix are the directions in which the data varies most (PCA, today). The top eigenvector of a link matrix is PageRank. Eigenvalues of the Hessian describe the curvature of the loss surface (why some learning rates diverge, Week 6). Eigenvalues of the weight matrices explain exploding and vanishing gradients (Week 14).

**By hand:** $Av = \lambda v \iff (A - \lambda I)v = 0$. A non-zero $v$ exists only if $A - \lambda I$ squashes space, so $\det(A - \lambda I) = 0$. For a 2×2 matrix this gives $\lambda^2 - \text{tr}(A)\lambda + \det(A) = 0$.

In [ ]:
A = np.array([[2.0, 1.0], [1.0, 2.0]])
vals, vecs = np.linalg.eigh(A)          # eigh is for symmetric matrices
print("eigenvalues", vals)
print("eigenvectors (columns)\n", vecs)
for i in range(2):
    v = vecs[:, i]
    print(f"A @ v{i} = {A @ v}  and  λ v{i} = {vals[i] * v}")
show_transform(A, "symmetric A")

✍️ *In the plot, find the two directions that only got stretched. Do they match the eigenvectors printed above? (They're at ±45°.) Now find the eigenvalues of [[2, 1], [1, 2]] by hand with the formula above, on paper.* (double-click to edit)



### Exercise 5.5: `eig2x2`
Return the two eigenvalues of a 2×2 matrix (largest first) using the trace/determinant quadratic formula. You may assume they're real.

In [ ]:
def eig2x2(A):
    raise NotImplementedError

In [ ]:
for M in [np.array([[2.0, 1.0], [1.0, 2.0]]), np.array([[4.0, 1.0], [2.0, 3.0]]), np.array([[1.0, 0.0], [0.0, -2.0]])]:
    got = eig2x2(M)
    expected = sorted(np.linalg.eigvals(M).real, reverse=True)
    assert np.allclose(got, expected), (got, expected)
print("✅ 5.5 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

λ = (tr ± √(tr² − 4·det)) / 2, with `tr = A[0, 0] + A[1, 1]`.

</details>


### Exercise 5.6: Power iteration
If you multiply any starting vector by $A$ over and over (normalising each time), the component along the eigenvector with the **largest** |λ| grows fastest and eventually dominates. Implement `power_iteration(A, n_iter=200, seed=0)`, returning `(eigenvalue, eigenvector)`. Use the Rayleigh quotient $\lambda \approx v^\top A v$ for unit-length $v$.

This one idea underlies PageRank and many large-scale eigen-solvers.

In [ ]:
def power_iteration(A, n_iter=200, seed=0):
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(3)
M = rng.normal(size=(6, 6)); S = M @ M.T            # symmetric with positive eigenvalues
lam, v = power_iteration(S)
vals, vecs = np.linalg.eigh(S)
assert abs(lam - vals[-1]) < 1e-6
assert abs(abs(v @ vecs[:, -1]) - 1) < 1e-6         # same direction up to sign
print("✅ 5.6 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Start with `v = rng.normal(size=A.shape[0])` and normalise it.

</details>

<details><summary>Hint 2</summary>

Loop: `v = A @ v`, then `v = v / np.linalg.norm(v)`. After the loop, `lam = v @ A @ v`.

</details>


---
# Day 4: PCA from scratch (2 h)

**Principal Component Analysis** finds the directions of greatest variance:
1. **Centre** the data: $X_c = X - \bar{x}$.
2. **Covariance**: $C = \frac{1}{n-1} X_c^\top X_c$, a symmetric $d \times d$ matrix.
3. **Eigen-decompose** $C$. The eigenvectors are the *principal components* and the eigenvalues are the *variance along each one*.
4. **Project**: $Z = X_c W_k$ with the top $k$ eigenvectors as columns of $W_k$.

The dataset is scikit-learn's bundled **Breast Cancer Wisconsin (Diagnostic)** set: 569 tumours, 30 features computed from images of cell nuclei, each labelled malignant or benign. It's public, de-identified, and ships with scikit-learn.

In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X, y = data.data, data.target
print(X.shape, "features e.g.:", list(data.feature_names[:4]))
print("feature scales differ hugely:", X.std(axis=0)[:4])
X_std = (X - X.mean(axis=0)) / X.std(axis=0)     # standardise, so no feature dominates by units alone

### Exercise 5.7: `pca`
Return `(components, explained_variance_ratio, Z)`:
- `components`: a (k, d) array with one principal direction per **row**, sorted by variance, largest first,
- `explained_variance_ratio`: a length-k array of each component's share of the total variance,
- `Z`: the (n, k) projected data.

In [ ]:
def pca(X, k):
    raise NotImplementedError

In [ ]:
from sklearn.decomposition import PCA
comps, evr, Z = pca(X_std, 2)
ref = PCA(n_components=2).fit(X_std)
assert comps.shape == (2, 30) and Z.shape == (569, 2)
assert np.allclose(evr, ref.explained_variance_ratio_)
for i in range(2):
    assert abs(abs(comps[i] @ ref.components_[i]) - 1) < 1e-6   # equal up to sign
plt.scatter(Z[:, 0], Z[:, 1], c=y, cmap="coolwarm", s=8)
plt.xlabel(f"PC1 ({evr[0]:.0%})"); plt.ylabel(f"PC2 ({evr[1]:.0%})"); plt.title("Breast cancer: 30 features → 2"); plt.show()
print("✅ 5.7 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`Xc = X - X.mean(axis=0)`, then `C = Xc.T @ Xc / (len(X) - 1)`.

</details>

<details><summary>Hint 2</summary>

`vals, vecs = np.linalg.eigh(C)` returns them in **ascending** order. Reverse with `[::-1]`.

</details>

<details><summary>Hint 3</summary>

`components = vecs[:, order[:k]].T`. The ratio is `vals[order[:k]] / vals.sum()`, and `Z = Xc @ components.T`.

</details>


✍️ *Two components explain what fraction of the variance? Looking at the plot, would a simple linear classifier do well on this data? What does PCA *not* use that a classifier would?* (double-click to edit)



---
# Review (30 min)

### Blank-page rebuild
Rewrite `cosine_matrix` from memory.

In [ ]:
def cosine_matrix2(A, B):
    raise NotImplementedError

rng = np.random.default_rng(9)
A, B = rng.normal(size=(5, 3)), rng.normal(size=(4, 3))
assert np.allclose(cosine_matrix2(A, B), cosine_matrix(A, B))
print("✅ rebuild passed")

### Spaced recall
1. *(Week 2)* What is a dict comprehension? Write one that maps each word to its length.
2. *(Week 3)* Why is `x in set` O(1)?
3. *(Week 4)* What is an inverted index, and why is it faster than scanning every document?
4. What does the determinant tell you geometrically? What does $\det = 0$ mean?
5. Explain eigenvectors to a first-year student in two sentences, without formulas.
6. Why do we standardise features before PCA?
7. Why is `A @ B` so much faster than three Python loops?

✍️ *Your answer:* (double-click to edit)



### Commit, log, update the tracker
Use a `week05` branch, a PR, and a merge. Update `LOG.md` and the **Mathematics** rows in `COMPETENCY_TRACKER.md`.